### JSON 형식 출력 파서 ( JsonOutputParser )

- 답변으로 단순한 문자열이 아니라 지정된 스키마에 맞게 JSON 형식으로 데이터 반환
- 사용자가 원하는 형태의 JSON을 생성하기 위해 모델의 용량이 충분히 커야 함
- 용량이 작은 모델에서는 오류가 발생할 수 있음

In [4]:
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

# 모델
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 프롬프트
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    FewShotPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

# 예시 선택기 / 벡터스토어
from langchain_core.example_selectors import (
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector,
)
from langchain_chroma import Chroma

# 출력 파서
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser, JsonOutputParser
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

# teddynote 유틸
from langchain_teddynote import logging
from langchain_teddynote.messages import stream_response


# ── 환경 설정 ──────────────────────────────
load_dotenv()
logging.langsmith("test0914")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914


In [2]:
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtag: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [6]:
question= "지구 온난화의 심각성에 대해 알려주세요."

parser = JsonOutputParser(pydantic_object = Topic)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [7]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요"),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}")
    ]
)


prompt = prompt.partial(format_instructions = parser.get_format_instructions())

chain = prompt | llm | parser

answer = chain.invoke({"question": question})

In [8]:
answer["description"]

'지구 온난화는 지구의 평균 기온이 상승하는 현상으로, 기후 변화, 해수면 상승, 생태계 파괴 등 심각한 영향을 미칩니다.'